# [1장 4강] - 실습: 특수 행렬과 행렬 연산 성질

**실습 목표**

- 단위행렬·대각행렬·대칭행렬의 성질을 코드로 확인하고 용도를 설명할 수 있다.
- 행렬 곱이 교환법칙을 만족하지 않음을 예제로 검증할 수 있다.
- 전치의 성질 `(AB)ᵀ = BᵀAᵀ`를 이용해 `XᵀX`가 항상 대칭임을 설명할 수 있다.
- 행렬식으로 역행렬 존재 여부를 판단하고, 행렬식이 0이 되거나 조건수가 급격히 커질 때의 위험을 설명할 수 있다.

**실습에 필요한 데이터셋/파일**

- 분야: 금융
- 데이터셋: UCI Default of Credit Card Clients
- 사용 방식: `ucimlrepo.fetch_ucirepo(id=350)`
- 출처: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
- 사용 목적: 신용카드 고객의 수치형 특성으로 데이터 행렬을 만들어 대칭행렬(`XᵀX`)과 행렬식을 확인합니다.
- 준비물: Python, NumPy, pandas, scikit-learn, ucimlrepo

- 이번 강의의 성질(단위행렬, 비교환성, 전치)은 **작은 행렬로 확인하는 편이 훨씬 명확합니다.** 실제 데이터는 대칭행렬과 행렬식을 확인할 때만 사용하고, 나머지는 2×2 예제 행렬로 검증합니다.
- 실데이터로 열별 표준편차를 비교할 때는 **행을 충분히(최소 50행) 확보**하세요. `EDUCATION`처럼 값의 종류가 적은 컬럼은 앞 5행만 자르면 모든 값이 같아져 표준편차가 0이 되고, 표준화한 값에서는 `nan`이 나올 수 있습니다.
- 행렬식과 역행렬은 **정사각행렬에만 정의**됩니다. 데이터 행렬 `X`는 보통 정사각이 아니므로, `XᵀX` 형태로 만들어 정사각행렬을 확보합니다.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def load_uci(dataset_id):
    """UCI에서 데이터를 불러오고, 실패하면 구조가 비슷한 대체 데이터를 사용합니다."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=dataset_id)
        X, y = ds.data.features.copy(), ds.data.targets.copy()
        if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
            y = y.iloc[:, 0]
        return X, y
    except Exception as e:
        print('[안내] UCI 로드 실패:', e)
        print('[안내] 대체 데이터로 진행합니다. 성질 검증 방법은 동일합니다.')
        from sklearn.datasets import make_classification
        Xa, ya = make_classification(n_samples=2500, n_features=20,
                                     n_informative=8, random_state=RANDOM_STATE)
        cols = [f'feature_{i}' for i in range(Xa.shape[1])]
        return pd.DataFrame(Xa, columns=cols), pd.Series(ya)


def numeric_frame(X):
    """수치형 컬럼만 남기고 결측값을 중앙값으로 채웁니다."""
    Xn = X.select_dtypes(include='number').copy()
    Xn = Xn.replace([np.inf, -np.inf], np.nan)
    return Xn.fillna(Xn.median(numeric_only=True))


X_raw, y_raw = load_uci(350)   # Default of Credit Card Clients
X = StandardScaler().fit_transform(numeric_frame(X_raw).iloc[:, :4])   # 특성 4개만 사용
print('데이터 행렬 X:', X.shape)

데이터 행렬 X: (30000, 4)


## 필수 1 : 특수 행렬로 데이터 축을 조정하기
리스크 분석팀은 신용 점수 모델에서 특정 항목(예: 연체 이력)의 비중을 키우고 다른 항목의 비중을 줄이는 실험을 하려 합니다. 이때 항목별 가중치를 대각행렬로 표현하면 행렬곱 한 번으로 축별 비중을 조정할 수 있습니다. 먼저 "곱해도 아무것도 바꾸지 않는" 단위행렬을 기준점으로 두고, 대각행렬과 대칭행렬의 성질을 확인합니다.

### 문제 1-1 : 단위행렬과 대각행렬 적용하기
1. `X`의 **앞 50행**을 `A`로 두고, `A`의 열 수에 맞는 단위행렬 `I`를 만듭니다. (앞 5행처럼 너무 적게 자르면 `EDUCATION`같이 값의 종류가 적은 컬럼에서 모든 값이 같아져 표준편차가 0이 되고, 4번의 비교가 성립하지 않습니다.)
2. `A @ I`가 `A`와 같은지 `np.allclose`로 확인합니다.
3. 항목별 비중을 `[1, 10, 0.1, 2]`로 조정하는 대각행렬 `D`를 만들고 `A @ D`를 계산합니다.
4. `A`와 `A @ D`의 각 열 표준편차를 비교해, 대각행렬이 어떤 축에 어떤 영향을 주었는지 설명합니다. 표준편차가 0인 열이 있으면 표본 행 수를 더 늘려 다시 확인합니다.

In [2]:
# 1. `X`의 **앞 50행**을 `A`로 두고, `A`의 열 수에 맞는 단위행렬 `I`를 만듭니다. (앞 5행처럼 너무 적게 자르면 `EDUCATION`같이 값의 종류가 적은 컬럼에서 모든 값이 같아져 표준편차가 0이 되고, 4번의 비교가 성립하지 않습니다.)
A = X[0:50]
print(A.shape)
I = np.eye(4)
print(I.shape)

# 2. `A @ I`가 `A`와 같은지 `np.allclose`로 확인합니다.
print(np.allclose(A, A @ I))

# 3. 항목별 비중을 `[1, 10, 0.1, 2]`로 조정하는 대각행렬 `D`를 만들고 `A @ D`를 계산합니다.
D = np.diag([1, 10, 0.1, 2])
# print(D)
A_D = A @ D

# 4. `A`와 `A @ D`의 각 열 표준편차를 비교해, 대각행렬이 어떤 축에 어떤 영향을 주었는지 설명합니다. 표준편차가 0인 열이 있으면 표본 행 수를 더 늘려 다시 확인합니다.
std_A = A.std(axis=0)
std_A_D = A_D.std(axis=0)
print(std_A)
print(std_A_D)
# 표준편차도 각 배율만큼 바뀌었다.



(50, 4)
(4, 4)
True
[1.18574631 1.02142373 1.09752118 0.90350263]
[ 1.18574631 10.21423728  0.10975212  1.80700525]


### 문제 1-2 : 데이터에서 대칭행렬 찾기
1. `G = X.T @ X`를 계산하고 shape을 확인합니다.
2. `G`가 자기 자신의 전치와 같은지(`G == G.T`) 확인합니다.
3. `np.cov(X, rowvar=False)`로 공분산행렬을 만들고 마찬가지로 대칭인지 확인합니다.
4. 공분산행렬이 대칭일 수밖에 없는 이유를 "변수 i와 j의 공분산" 관점에서 한 문장으로 설명합니다.

In [ ]:
# 1. `G = X.T @ X`를 계산하고 shape을 확인합니다.
X.shape # (30000, 4)
X.T.shape # (4, 30000)
G = X.T @ X # (4, 4) 특성 데이터만 남김
# G2 = X @ X.T # (30000, 30000)

# 2. `G`가 자기 자신의 전치와 같은지(`G == G.T`) 확인합니다.
print(np.allclose(G, G.T))

# 3. `np.cov(X, rowvar=False)`로 공분산행렬을 만들고 마찬가지로 대칭인지 확인합니다.
cov = np.cov(X, rowvar=False)
cov

# 4. 공분산행렬이 대칭일 수밖에 없는 이유를 "변수 i와 j의 공분산" 관점에서 한 문장으로 설명합니다.
# 공분산 행렬의 (i, j) 원소는 변수 i와 변수 i의 공분산 Cov(i, j)이고,
# (j, i) 원소는 변수 j와 변수 i의 공분산 Cov(j, i)인데,
# 곱셈의 교환법칙 때문에 이 둘이 같으므로 행렬이 대칭이다.

True


array([[ 1.00003333,  0.02475606, -0.219168  , -0.10814302],
       [ 0.02475606,  1.00003333,  0.01423241, -0.03138989],
       [-0.219168  ,  0.01423241,  1.00003333, -0.14346912],
       [-0.10814302, -0.03138989, -0.14346912,  1.00003333]])

## 필수 2 : 연산 순서를 바꾸면 결과가 달라진다
리스크 모델에서는 "표준화 후 가중치 적용"과 "가중치 적용 후 표준화"처럼 처리 순서를 바꿔 실험하는 일이 잦습니다. 숫자의 곱셈과 달리 행렬 곱은 순서를 바꾸면 결과가 달라지므로, 순서를 바꾼 실험은 다른 처리로 취급해야 합니다. 이 성질을 직접 확인하고, 전치를 취할 때 순서가 뒤집히는 이유까지 이어서 살펴봅니다.

### 문제 2-1 : AB ≠ BA 확인하기
1. 2×2 행렬 `P = [[1, 2], [0, 1]]`, `Q = [[1, 0], [3, 1]]`을 정의합니다.
2. `P @ Q`와 `Q @ P`를 각각 계산해 출력합니다.
3. 두 결과가 같은지 `np.array_equal`로 확인합니다.
4. 이번에는 `P`를 **2 × 단위행렬**(`2 * np.eye(2)`)로 바꿔 `Q`와의 곱을 다시 비교합니다. 단위행렬의 스칼라배는 어떤 행렬과도 교환됩니다.
5. 비교를 위해 대각 원소가 서로 다른 대각행렬 `np.diag([2, 5])`도 `Q`와 곱해 보고, **이 경우에는 순서를 바꾸면 결과가 달라진다**는 것을 확인합니다. 즉 "대각행렬이면 항상 교환된다"는 결론은 성립하지 않습니다. (대각행렬끼리 곱할 때만 순서가 무관합니다.)
6. 행렬 곱이 교환법칙을 만족하지 않는 이유를 "행과 열의 내적" 관점에서 설명합니다.

In [ ]:
# 1. 2×2 행렬 `P = [[1, 2], [0, 1]]`, `Q = [[1, 0], [3, 1]]`을 정의합니다.\
P = np.array([
    [1, 2],
    [0, 1]
    ])
Q = np.array([
    [1, 0],
    [3, 1]
    ])

# 2. `P @ Q`와 `Q @ P`를 각각 계산해 출력합니다.
P_Q = P @ Q
Q_P = Q @ P
print(P_Q)
print(Q_P)

# 3. 두 결과가 같은지 `np.array_equal`로 확인합니다.
print(np.array_equal(P_Q, Q_P))

# 4. 이번에는 `P`를 **2 × 단위행렬**(`2 * np.eye(2)`)로 바꿔 `Q`와의 곱을 다시 비교합니다. 단위행렬의 스칼라배는 어떤 행렬과도 교환됩니다.
P = np.eye(2) * 2
P
print(P @ Q)
print(Q @ P)


# 5. 비교를 위해 대각 원소가 서로 다른 대각행렬 `np.diag([2, 5])`도 `Q`와 곱해 보고, **이 경우에는 순서를 바꾸면 결과가 달라진다**는 것을 확인합니다. 즉 "대각행렬이면 항상 교환된다"는 결론은 성립하지 않습니다. (대각행렬끼리 곱할 때만 순서가 무관합니다.)
diag = np.diag([2, 5])
print(diag @ Q)
print(Q @ diag)

# 6. 행렬 곱이 교환법칙을 만족하지 않는 이유를 "행과 열의 내적" 관점에서 설명합니다.
# i번째 행과 j번째 열의 계산이기 때문에 순서를 뒤집는 다면 행과 열의 게산이 틀려져 결과도 다르게 나온다.
# 순서를 뒤집으면 행을 제공하는 행렬과 열을 제공하는 행렬이 서로 바뀌어 내적하는 벡터 자체가 달라지므로, 계산 결과도 달라진다

[[7 2]
 [3 1]]
[[1 2]
 [3 7]]
False
[[2. 0.]
 [6. 2.]]
[[2. 0.]
 [6. 2.]]
[[ 2  0]
 [15  5]]
[[2 0]
 [6 5]]


### 문제 2-2 : 전치 성질로 XᵀX가 대칭인 이유 설명하기
1. `(P @ Q).T`와 `Q.T @ P.T`를 계산해 값이 같은지 확인합니다.
2. 순서를 유지한 `P.T @ Q.T`도 계산해 값이 다른지 확인합니다.
3. `(XᵀX)ᵀ`를 전치 성질로 전개하면 다시 `XᵀX`가 됨을 식으로 적고, 코드로도 확인합니다.
4. 이 성질이 왜 "공분산행렬은 항상 대칭"이라는 결론으로 이어지는지 한 문장으로 작성합니다.

In [ ]:
# 1. `(P @ Q).T`와 `Q.T @ P.T`를 계산해 값이 같은지 확인합니다.
P = np.array([
    [1, 2],
    [0, 1]
    ])
P
Q
PQ_T = (P @ Q).T
QT_PT = Q.T @ P.T
print(np.array_equal(PQ_T, QT_PT))

# 2. 순서를 유지한 `P.T @ Q.T`도 계산해 값이 다른지 확인합니다.
PT_QT = P.T @ Q.T
print(PT_QT)
print(np.array_equal(PQ_T, PT_QT))

# 3. `(XᵀX)ᵀ`를 전치 성질로 전개하면 다시 `XᵀX`가 됨을 식으로 적고, 코드로도 확인합니다.
# (XᵀX)ᵀ = Xᵀ @ (Xᵀ)ᵀ = Xᵀ @ X = XᵀX
XTX = X.T @ X
XTX_T = (X.T @ X).T
print(np.array_equal(XTX, XTX_T))

# 4. 이 성질이 왜 "공분산행렬은 항상 대칭"이라는 결론으로 이어지는지 한 문장으로 작성합니다.
# 전치해도 자기 자신인 행렬을 대칭이라고 하고, 중심화한 X.T@X인 공분산행렬은 대칭이다.


True
[[1 3]
 [2 7]]
False
True


## 심화 1 : 역행렬을 믿을 수 있는지 행렬식과 조건수로 판단하기
리스크 모델을 학습할 때 XᵀX의 역행렬이 필요한 경우가 있습니다. 그런데 데이터에 사실상 중복인 컬럼(예: 월 소득과 연 소득)이 섞여 있으면 역행렬 계산이 불안정해지고, 계수 값이 비정상적으로 커지거나 실행할 때마다 달라집니다. 행렬식과 조건수를 함께 확인해 이 위험을 미리 감지합니다.

### 문제 3-1 : 행렬식과 조건수로 역행렬 존재·안정성 확인하기
1. `A = [[1, 2], [3, 4]]`의 행렬식을 계산하고, 역행렬을 구해 `A @ A_inv ≈ I`인지 확인합니다.
2. `S = [[1, 2], [2, 4]]`(두 번째 행이 첫 행의 2배)의 행렬식을 계산하고, 역행렬을 시도해 어떤 오류가 나는지 확인합니다.
3. 실제 데이터 `X`에 **첫 번째 컬럼의 2배인 중복 컬럼**을 추가한 `X_dup`을 만듭니다.
4. `XᵀX`와 `X_dupᵀX_dup`의 행렬식과 조건수(`np.linalg.cond`)를 비교합니다.
5. 행렬식이 0이 되거나 조건수가 급격히 커질 때 어떤 문제가 생기는지, 실무에서 어떻게 대응해야 하는지 2~3문장으로 작성합니다. 이때 **행렬식의 절댓값 크기 자체는 판단 기준이 되지 않는다**는 점도 함께 정리합니다.

In [ ]:
# 1. `A = [[1, 2], [3, 4]]`의 행렬식을 계산하고, 역행렬을 구해 `A @ A_inv ≈ I`인지 확인합니다.
A = np.array([
    [1, 2],
    [3, 4]
])

A_det = np.linalg.det(A)
print(A_det)

A_inv = np.linalg.inv(A)
print(A_inv)
print(A @ A_inv)

# 2. `S = [[1, 2], [2, 4]]`(두 번째 행이 첫 행의 2배)의 행렬식을 계산하고, 역행렬을 시도해 어떤 오류가 나는지 확인합니다.
S = np.array([
    [1, 2],
    [2, 4]
])

S_det = np.linalg.det(S)
print(S_det)

try:
    S_inv = np.linalg.inv(S)
except Exception as e:
    print(e)


# 3. 실제 데이터 `X`에 **첫 번째 컬럼의 2배인 중복 컬럼**을 추가한 `X_dup`을 만듭니다.
# 첫 번째 열을 꺼내서 2배 → 세로 모양으로
dup_col = (X[:, 0] * 2).reshape(-1, 1)
# 원래 X 오른쪽에 그 열을 나란히 붙이기
X_dup = np.hstack([X, dup_col])
print(X_dup)

# 4. `XᵀX`와 `X_dupᵀX_dup`의 행렬식과 조건수(`np.linalg.cond`)를 비교합니다.
XTX = X.T @ X
X_dupTX = X_dup.T @ X_dup
# print(XTX)
# print(X_dupTX)
det_1 = np.linalg.det(XTX)
det_2 = np.linalg.det(X_dupTX)
print(det_1, det_2)

cond_1 = np.linalg.cond(XTX)
cond_2 = np.linalg.cond(X_dupTX)
print(cond_1, cond_2)

# 5. 행렬식이 0이 되거나 조건수가 급격히 커질 때 어떤 문제가 생기는지, 실무에서 어떻게 대응해야 하는지 2~3문장으로 작성합니다. 이때 **행렬식의 절댓값 크기 자체는 판단 기준이 되지 않는다**는 점도 함께 정리합니다.
# 행렬식이 0이 되면 역행렬을 구하지 못한다.
# 조건수가 크면 역이 수치적으로 불안정해진다.
# 중복되거나 상관 된 특성을 선별하여 제거한다.


-2.0000000000000004
[[-2.   1. ]
 [ 1.5 -0.5]]
[[1.0000000e+00 0.0000000e+00]
 [8.8817842e-16 1.0000000e+00]]
0.0
Singular matrix
[[-1.13672015  0.81016074  0.18582826 -1.05729503 -2.27344029]
 [-0.3659805   0.81016074  0.18582826  0.85855728 -0.731961  ]
 [-0.59720239  0.81016074  0.18582826  0.85855728 -1.19440479]
 ...
 [-1.05964618 -1.23432296  0.18582826  0.85855728 -2.11929236]
 [-0.67427636 -1.23432296  1.45111372 -1.05729503 -1.34855272]
 [-0.90549825 -1.23432296  0.18582826 -1.05729503 -1.8109965 ]]
7.381974247534394e+17 0.0
1.7963916161096218 1.985092970877837e+16
